In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

torch.cuda.is_available()

d:\a-study-on-transformer\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


True

In [4]:
from package import create_dataloader, make_collate_fn

sequences = [[1,2,3,4,5,6,7], [10,20,30,40,50], [100,200,300]]

collate_fn = make_collate_fn(min_len=3, max_len=5)
loader = create_dataloader(sequences, batch_size=4, collate_fn=collate_fn)

# for x1_padded, x1_mask, x2_padded, x2_mask in loader:
#     # x1_padded, x2_padded: (batch, max_crop_len) — the positive pairs
#     # x1_mask, x2_mask: (batch, max_crop_len) — True where real tokens exist
#     pass


In [40]:
next(iter(loader))

(tensor([[100, 200, 300,   0,   0],
         [  3,   4,   5,   6,   7],
         [ 10,  20,  30,  40,  50]]),
 tensor([[ True,  True,  True, False, False],
         [ True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True]]),
 tensor([[100, 200, 300,   0],
         [  4,   5,   6,   7],
         [ 20,  30,  40,  50]]),
 tensor([[ True,  True,  True, False],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]]))

In [41]:
import random

import torch

from package import (
    ContrastiveTransformerEncoder,
    create_dataloader,
    make_collate_fn,
    nt_xent_loss,
)
from package.pad_and_mask import pad_and_mask

# --- Mock data: 100 sequences (vocab 1-50, length 5-20) ---
random.seed(42)
sequences = [
    [random.randint(1, 50) for _ in range(random.randint(5, 20))]
    for _ in range(100)
]

# --- Config ---
VOCAB_SIZE = 51  # 0 reserved for padding
BATCH_SIZE = 16
EPOCHS = 30
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Setup ---
collate_fn = make_collate_fn(min_len=3, max_len=10)
loader = create_dataloader(sequences, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
model = ContrastiveTransformerEncoder(vocab_size=VOCAB_SIZE).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [43]:
# --- Train ---
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for x1, mask1, x2, mask2 in loader:
        x1, mask1 = x1.to(DEVICE), mask1.to(DEVICE)
        x2, mask2 = x2.to(DEVICE), mask2.to(DEVICE)

        z1 = model(x1, mask1)
        z2 = model(x2, mask2)
        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss / len(loader):.4f}")

# --- Embed ---
model.eval()
padded, mask = pad_and_mask(sequences)
padded, mask = padded.to(DEVICE), mask.to(DEVICE)

with torch.no_grad():
    embeddings = model.encode(padded, mask)

print(f"Embeddings shape: {embeddings.shape}")  # (100, 64)


Epoch 5/30 — Loss: 0.6997
Epoch 10/30 — Loss: 1.1247
Epoch 15/30 — Loss: 1.0518
Epoch 20/30 — Loss: 0.6009
Epoch 25/30 — Loss: 0.7703
Epoch 30/30 — Loss: 0.6818
Embeddings shape: torch.Size([100, 64])


In [46]:
model.eval()
test_seq = [
    [1,2,3,4,5],
    [2,3,4,]
]

padded, mask = pad_and_mask(test_seq)
padded, mask = padded.to(DEVICE), mask.to(DEVICE)

with torch.no_grad():
    test_embeddings = model.encode(padded, mask)

In [47]:
test_embeddings.shape

torch.Size([2, 64])

In [ ]:
test_embeddings

tensor([[-0.6161,  0.2235,  0.3251, -0.3602, -0.6219, -0.3066, -0.7557, -0.3579,
         -0.4501,  0.1221, -0.0885, -0.1201, -0.0827,  0.1958,  0.7620,  0.5080,
          0.9234, -1.1917,  0.4670, -0.4017,  0.1231,  0.1340,  1.3401, -0.6038,
         -0.6314, -0.1980,  0.9402, -0.2230,  0.7980, -0.7609,  0.1037,  0.0702,
         -0.2732, -0.1612,  0.1278,  0.9268,  0.4600,  0.5683,  0.0841, -0.0990,
          0.1300,  0.1070,  0.1416, -0.4928, -0.3159, -0.9604,  0.0969, -0.1704,
          0.0026,  0.3135,  0.2799, -0.4487, -0.3264,  0.2885,  0.1499,  0.5353,
          0.0490, -0.1858,  0.0054, -1.0378,  0.4044,  0.2680,  0.5088, -0.1997],
        [-0.5189,  0.8505, -0.2916, -0.8638, -0.9325, -0.3451, -0.6409, -0.6226,
         -0.2953,  0.7332,  0.1995, -0.2840, -0.1851, -0.3154,  1.0509, -0.2560,
          1.0680, -1.5672,  0.5180, -0.2606,  0.1669,  0.5390,  1.2483, -0.8203,
         -1.0770, -0.3879,  1.1536, -0.5389,  0.8715, -0.2767,  0.2090,  0.3282,
          0.2418, -0.5702, 

In [49]:
# Cosine similarity (dot product on normalized vectors)
from torch.nn.functional import cosine_similarity

sim = cosine_similarity(test_embeddings[0].unsqueeze(0), test_embeddings[1].unsqueeze(0))
# Returns value in [-1, 1] — closer to 1 means more similar
sim


tensor([0.8082], device='cuda:0')